# Figure 2D - UMAPs and Graph dissimilarity analysis

In [ ]:
from pathlib import Path
import scanpy as sc
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from tqdm import tqdm
from pprint import pprint

import importlib
import scatlastb_utils as atl

sc.set_figure_params(frameon=False, dpi=80, fontsize=12, dpi_save=300)
plt.rcParams["svg.fonttype"] = "none"

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
def clean_prefixes(adata, prefix):
    adata.obs.columns = adata.obs.columns.str.replace(prefix, '')
    
    adata.obsm = {
        key.replace(prefix, ''): adata.obsm[key]
        for key in adata.obsm.keys()
    }
    
    adata.obsp = {
        key.replace(prefix, ''): adata.obsp[key]
        for key in adata.obsp.keys()
    }
    
    adata.uns = {
        key.replace(prefix, ''): adata.uns[key]
        for key in adata.uns.keys()
    }

In [ ]:
integrations = [
    'HLCAv1',
    'highQC',
]

In [ ]:
figure_dir = Path('figures/figure2/D/')
umap_dir = figure_dir / 'umaps'
umap_dir.mkdir(exist_ok=True, parents=True)

In [ ]:
gd_dir = figure_dir / 'graph_dissimilarity'
gd_dir.mkdir(exist_ok=True, parents=True)

## Read Data

In [ ]:
adata = atl.io.read_anndata(
    # 'data/pipeline/qc/marker_genes/dataset~map_extended/file_id~majority_voting:collect:map_extended.zarr',
    'data/pipeline/HLCAv1/QC/marker_genes/dataset~HLCAv1_extended/file_id~majority_voting:collect:HLCAv1_extended.zarr',
    X='X',
    obs='obs',
    var='var',
    obsm='obsm',
    uns='uns',
    obsp='obsp',
    # dask_slots=['layers', 'X', 'obsp'],
    dask=True,
    backed=True,
)
clean_prefixes(adata, 'label_transfer:clustering:')
adata

In [ ]:
adata.obs['n_genes'] = adata.obs['n_genes'].astype('float32')

In [ ]:
extended_only_mask = adata.obs['core_or_extension'] == 'extension'
core_only = adata.obs['core_or_extension'] == 'core'
highQC_reference = core_only & (adata.obs['qc_status'] == 'passed')
# extended_only_mask | adata.obs.eval('core_or_extension == "core" & qc_status == "passed"')

In [ ]:
# set colors
ref_key = 'ann_finest_level'
for col in tqdm([x for x in adata.obs.columns if x.startswith('majority_reference')]):
    adata.obs[col] = adata.obs[col].cat.set_categories(adata.obs[ref_key].cat.categories)

## UMAPs

In [ ]:
colors = [
    'core_or_extension',
    'ann_finest_level',
    # 'ann_level_1',
    'ann_level_2',
    'ann_level_3',
    # 'leiden_0.5_1',
    # 'transf_ann_level_5_uncert',
]

In [ ]:
qc_colors = [
    'qc_status',
    'scrublet_score',
    'n_genes',
    'n_counts',
    'percent_mito',
]
adata.obs['n_genes'] = adata.obs['n_genes'].astype('float32')

In [ ]:
show_on_extended_only = [
    'qc_status',
    'scrublet_score',
    'percent_mito',
]

In [ ]:
for color in colors:
    for embedding in integrations:
        atl.pl.embedding(
            adata,
            basis=f'X_umap--{embedding}',
            color=color,
            plot_centroids=color != 'core_or_extension',
            max_label_length=0,
            title=f'{color} - {embedding}',
            size=1,
            verbose=False,
            dpi=300,
        )
        plt.savefig(umap_dir / f'Ext2_C_{color}_{embedding}.svg')

In [ ]:
for color in tqdm(qc_colors):
    for embedding in integrations:
        ax = sc.pl.embedding(
            adata,
            basis=f'X_umap--{embedding}',
            color=color,
            title=f'{color} - {embedding} embedding',
            size=1,
            show=False,
        )
        ax.figure.savefig(umap_dir / f'Ext2_C_{color}_embedding.svg')
        plt.close(ax.figure)

        ax = sc.pl.embedding(
            adata,
            basis=f'X_umap--{embedding}',
            color=color,
            title=f'{color} - {embedding} embedding (extended only)',
            size=1,
            mask_obs=extended_only_mask,
            show=False,
        )
        ax.figure.savefig(umap_dir / f'Ext2_C_{color}_embedding_ext.svg')
        plt.close(ax.figure)
        
        ax = sc.pl.embedding(
            adata,
            basis=f'X_umap--{embedding}',
            color=color,
            title=f'{color} - {embedding} embedding (reference only)',
            size=1,
            mask_obs=highQC_reference,
            show=False,
        )
        ax.figure.savefig(umap_dir / f'Ext2_C_{color}_embedding_ref.svg')
        plt.close(ax.figure)

## Graph dissimilarity

In [ ]:
integration1, integration2 = integrations

In [ ]:
tqdm._instances.clear()

In [ ]:
%%time
comp = f'{integration1}-vs-{integration2}'

adata.obs[[
    f'avg_distance_1:{comp}',
    f'avg_distance_2:{comp}',
    f'avg_difference:{comp}',
    f'avg_distance_diff:{comp}',
    f'spearman_correlation:{comp}',
]] = atl.metrics.compare_distances(
    adata,
    obsp_connectivities_1=f'connectivities--{integration1}',
    obsp_distances_1=f'distances--{integration1}',
    obsm_key_1=f'X_emb--{integration1}',
    obsp_connectivities_2=f'connectivities--{integration2}',
    obsp_distances_2=f'distances--{integration2}',
    obsm_key_2=f'X_emb--{integration2}',
    scale_distances=True,
    log_scale_diffs=False,
    n_jobs=5,
)

comp = f'{integration2}-vs-{integration1}'
adata.obs[[
    f'avg_distance_1:{comp}',
    f'avg_distance_2:{comp}',
    f'avg_difference:{comp}',
    f'avg_distance_diff:{comp}',
    f'spearman_correlation:{comp}',
]] = atl.metrics.compare_distances(
    adata,
    obsp_connectivities_1=f'connectivities--{integration2}',
    obsp_distances_1=f'distances--{integration2}',
    obsm_key_1=f'X_emb--{integration2}',
    obsp_connectivities_2=f'connectivities--{integration1}',
    obsp_distances_2=f'distances--{integration1}',
    obsm_key_2=f'X_emb--{integration1}',
    scale_distances=True,
    log_scale_diffs=False,
    n_jobs=5,
)

In [ ]:
atl.me.plot_ranked_distances(adata, integration1, integration2)
atl.me.plot_distances_scatter(adata, integration1, integration2, s=1, alpha=0.5)
atl.me.plot_distances_scatter(adata, integration2, integration1, s=1, alpha=0.5)

### Ext Fig: 2C

In [ ]:
import matplotlib.pyplot as plt

def plot_graph_dissimilarity_umap(
    adata,
    integration1,
    integration2,
    metric,
    max_quantile=None,
    save=None,
    **kwargs,
):
    comparisons = [
        f'{integration1}-vs-{integration2}',
        f'{integration2}-vs-{integration1}'
    ]
    colors = [f'{metric}:{comparison}' for comparison in comparisons]
    ncols = len(comparisons)

    if max_quantile:
        kwargs['vmax'] = adata.obs[colors].quantile(max_quantile).tolist()
    kwargs['show'] = False

    sc.pl.embedding(
        adata,
        basis=f'X_umap--{integration1}',
        color=colors,
        ncols=ncols,
        wspace=0.2,
        **kwargs,
    )
    fig1 = plt.gcf()

    sc.pl.embedding(
        adata,
        basis=f'X_umap--{integration2}',
        color=colors,
        ncols=ncols,
        wspace=0.2,
        **kwargs,
    )
    fig2 = plt.gcf()

    if save:
        fig1.savefig(f'{save}_{integration1}.svg')
        fig2.savefig(f'{save}_{integration2}.svg')
    fig1.show()
    fig2.show()

In [ ]:
plot_graph_dissimilarity_umap(
    adata,
    integration1,
    integration2,
    metric='avg_distance_diff',
    cmap='viridis',
    size=2,
    vmin=0,
    max_quantile=0.999,
    mask_obs=adata.obs.eval('qc_status == "failed"'),
    save=gd_dir / 'Ext2_C',
)

In [ ]:
plot_graph_dissimilarity_umap(
    adata,
    integration1,
    integration2,
    metric='spearman_correlation',
    size=5,
    cmap="RdBu_r",
    sort_order=False,
    vcenter=0,
    mask_obs=adata.obs.eval('qc_status == "failed"')
)

In [ ]:
from misc import plot_diff_per_clusters

In [ ]:
plot_diff_per_clusters(
    adata,
    'qc_status',
    f'avg_distance_diff:{integration1}-vs-{integration2}',
)

In [ ]:
plot_diff_per_clusters(
    adata,
    'ann_level_2',
    f'avg_distance_diff:{integration1}-vs-{integration2}',
)

In [ ]:
color = "ann_finest_level"

order = (
    adata.obs
        .groupby(
            color,
            observed=True,
        )[f"avg_distance_diff:{integration2}-vs-{integration1}"]
        .quantile(0.99)
        .sort_values(ascending=False)
        .index
)
sns.catplot(
    adata.obs,
    y=color,
    x=f"avg_distance_diff:{integration2}-vs-{integration1}",
    kind="boxen",
    aspect=1,
    height=10,
    order=order,
)


### Ext Fig 2 D: Top dissimilar cell types

In [ ]:
color = 'ann_finest_level'
mask = adata.obs[color].isin(order[:5])
# mask = adata.obs[color].str.contains('DC').fillna(False)
# mask |= adata.obs[color].isin(['Ionocyte', 'CD8 T cells', 'B cells'])
adata.obs['color'] = np.where(mask, adata.obs[color].astype(str), float('nan'))
sc.pl.embedding(
    adata,
    basis='X_umap--HLCAv1',
    color='color',
    mask_obs=mask,
    size=5,
    palette=sc.pl.palettes.default_20,
)

fig = sc.pl.embedding(
    adata,
    basis='X_umap--highQC',
    color='color',
    mask_obs=mask,
    size=5,
    palette=sc.pl.palettes.default_20,
    return_fig=True,
)
fig.savefig(gd_dir / 'Ext2_D_umap_highQC_dissim_cts.svg')

### Fig 2D: Visualise low QC cluster

In [ ]:
color = 'leiden_0.5_3--highQC'
order = (
    adata.obs
        .groupby(
            color,
            observed=True,
        )[f"avg_distance_diff:{integration2}-vs-{integration1}"]
        .quantile(0.5)
        .sort_values(ascending=False)
        .index
)[:20]
g = sns.catplot(
    adata.obs.query(f'`{color}`.isin(@order)'),
    y=color,
    x=f"avg_distance_diff:{integration2}-vs-{integration1}",
    kind="boxen",
    aspect=0.8,
    height=5,
    order=order,
)
g.savefig(gd_dir / 'low_qc_cluster.svg')

In [ ]:
cluster_col = 'leiden_0.5_3--highQC'
cluster_of_interest = '29_1_3'

In [ ]:
# mask = adata.obs[cluster_col].isin(order[:1])
mask = adata.obs[cluster_col] == cluster_of_interest
adata.obs['color'] = np.where(mask, adata.obs[cluster_col].astype(str), float('nan'))

ax = sc.pl.embedding(
    adata,
    basis='X_umap--highQC',
    color='color',
    mask_obs=mask,
    size=10,
    palette=sc.pl.palettes.default_20,
    ncols=1,
    # verbose=False,
    show=False,
)
ax.figure.savefig(gd_dir / '2D_lowqc_cluster_highQC.svg', bbox_inches='tight')

In [ ]:
ax = sc.pl.embedding(
    adata,
    basis='X_umap--HLCAv1',
    color=['color'],
    mask_obs=mask,
    size=10,
    palette=sc.pl.palettes.default_20,
    # verbose=False,
    ncols=1,
    show=False,
)
ax.figure.savefig(gd_dir / '2D_lowqc_cluster_HLCAv1.svg', bbox_inches='tight')

## Extended Figure 2D: DEG for cluster of interest

In [ ]:
from tqdm import tqdm as _tqdm
import tqdm.notebook
tqdm.notebook.tqdm = _tqdm   # route notebook tqdm to the text version

In [ ]:
cluster_col = 'leiden_0.5_3--highQC'
cluster_of_interest = '29_1_3'
cts = ['Migratory DCs'] 

In [ ]:
adata_lowqc = adata[
    # adata.obs['ann_finest_level'].isin(cts) |
    (
        (adata.obs['ann_level_2'] == "Myeloid") &
        (adata.obs['ann_level_3'] != "Unknown")
    ) |
    (adata.obs[cluster_col] == cluster_of_interest)
].copy()

In [ ]:
adata_lowqc.obs[cluster_col] = adata_lowqc.obs[cluster_col].astype(str).where(
    adata_lowqc.obs[cluster_col] == cluster_of_interest,
    other='other',
)
adata_lowqc.obs['ann_finest_level'] = adata_lowqc.obs['ann_finest_level'].astype(str).where(
    adata_lowqc.obs['ann_finest_level'].isin(cts),
    other='other',
)
adata_lowqc.obs['deg_col'] = adata_lowqc.obs[
    ['ann_finest_level', cluster_col]
].astype(str).agg('-'.join, axis=1)

In [ ]:
adata.obs.query(f'(`{cluster_col}` == @cluster_of_interest)').shape

In [ ]:
adata.obs.query(
    f'(`{cluster_col}` == @cluster_of_interest) and (study == "Meyer_2021")'
)['ann_level_3'].astype(str).value_counts()

In [ ]:
adata.obs.query(f'(`{cluster_col}` == @cluster_of_interest)')['ann_finest_level'].astype(str).value_counts()

In [ ]:
adata_lowqc.obs['deg_col'].value_counts()

In [ ]:
adata_lowqc = atl.utils.dask_compute(adata_lowqc, layers='X')

In [ ]:
adata_lowqc.obs['ann_finest_level'].value_counts()

In [ ]:
adata_lowqc.obs[cluster_col].value_counts()

In [ ]:
adata_lowqc.obs['deg_col'].value_counts()

In [ ]:
adata_lowqc.var_names = adata_lowqc.var['feature_name'].astype(str)
adata_lowqc.var_names_make_unique()

In [ ]:
%%time 
sc.tl.rank_genes_groups(
    adata_lowqc,
    groupby='deg_col',
    reference='rest',
)

In [ ]:
sc.pl.rank_genes_groups(
    adata_lowqc,
    gene_symbols='feature_name',
    n_genes=10,
    ncols=2,
    show=False,
)
fig = plt.gcf()
fig.set_size_inches(7, 7)

for ax in fig.axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(
    gd_dir / 'Ext2_E_lowqc_cluster_marker_genes.svg',
    bbox_inches='tight',
)
plt.show()

In [ ]:
sc.tl.dendrogram(
    adata_lowqc,
    groupby='deg_col',
    use_rep='X_emb--highQC',
)

In [ ]:
fig = sc.pl.rank_genes_groups_dotplot(
    adata_lowqc,
    # gene_symbols='feature_name',
    n_genes=10,
    groups=[
        'Migratory DCs-29_1_3',
        'Migratory DCs-other',
        'other-other',
        # 'other-29_1_3',
    ],
    values_to_plot="logfoldchanges",
    cmap='bwr',
    vmin=-4,
    vmax=4,
    # min_logfoldchange=3,
    colorbar_title='log fold change',
    return_fig=True,
)
fig.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5)
fig.savefig(gd_dir / 'Ext2_E_lowqc_cluster_marker_genes_dotplot.svg')
plt.show()

In [ ]:
sc.pl.rank_genes_groups_violin(
    adata_lowqc,
    # gene_symbols='feature_name',
    n_genes=10,
    groups=['Migratory DCs-29_1_3', 'Migratory DCs-other'],
    return_fig=True
)